In [1]:
#Goal: Create a model that can predict the profit of the company based on company's spending
# pattern and company's location

In [2]:
import pandas as pd 
import numpy as np

In [3]:
np.set_printoptions(suppress=True)

In [4]:
data = pd.read_csv('50_Startups.csv')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [6]:
data['State'].unique()

array(['New York', 'California', 'Florida'], dtype=object)

In [7]:
# seperate feature and label column
features = data.iloc[:,[0,1,2,3]].values
label = data.iloc[:,4]

In [8]:
# state encoding
from sklearn.preprocessing import OneHotEncoder

oheState = OneHotEncoder(sparse_output=False)
stateDummy = oheState.fit_transform(features[:,3].reshape(-1,1))

In [9]:
finalFeatureSet = np.concatenate((stateDummy,features[:,[0,1,2]]),axis=1)
finalFeatureSet

array([[0.0, 0.0, 1.0, 165349.2, 136897.8, 471784.1],
       [1.0, 0.0, 0.0, 162597.7, 151377.59, 443898.53],
       [0.0, 1.0, 0.0, 153441.51, 101145.55, 407934.54],
       [0.0, 0.0, 1.0, 144372.41, 118671.85, 383199.62],
       [0.0, 1.0, 0.0, 142107.34, 91391.77, 366168.42],
       [0.0, 0.0, 1.0, 131876.9, 99814.71, 362861.36],
       [1.0, 0.0, 0.0, 134615.46, 147198.87, 127716.82],
       [0.0, 1.0, 0.0, 130298.13, 145530.06, 323876.68],
       [0.0, 0.0, 1.0, 120542.52, 148718.95, 311613.29],
       [1.0, 0.0, 0.0, 123334.88, 108679.17, 304981.62],
       [0.0, 1.0, 0.0, 101913.08, 110594.11, 229160.95],
       [1.0, 0.0, 0.0, 100671.96, 91790.61, 249744.55],
       [0.0, 1.0, 0.0, 93863.75, 127320.38, 249839.44],
       [1.0, 0.0, 0.0, 91992.39, 135495.07, 252664.93],
       [0.0, 1.0, 0.0, 119943.24, 156547.42, 256512.92],
       [0.0, 0.0, 1.0, 114523.61, 122616.84, 261776.23],
       [1.0, 0.0, 0.0, 78013.11, 121597.55, 264346.06],
       [0.0, 0.0, 1.0, 94657.16, 145077.58

# 1. Correlation Analysis

In [10]:
# extract correlation

finalDataSetDF = pd.concat([pd.get_dummies(data.State,dtype=int),data.iloc[:,[0,1,2,4]]],axis=1)
finalDataSetDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   California       50 non-null     int64  
 1   Florida          50 non-null     int64  
 2   New York         50 non-null     int64  
 3   R&D Spend        50 non-null     float64
 4   Administration   50 non-null     float64
 5   Marketing Spend  50 non-null     float64
 6   Profit           50 non-null     float64
dtypes: float64(4), int64(3)
memory usage: 2.9 KB


In [11]:
finalDataSetDF.corr()

,California,Florida,New York,R&D Spend,Administration,Marketing Spend,Profit
California,1.000000,-0.492366,-0.515152,-0.143165,-0.015478,-0.168875,-0.145837
Florida,-0.492366,1.000000,-0.492366,0.105711,0.010493,0.205685,0.116244
New York,-0.515152,-0.492366,1.000000,0.039068,0.005145,-0.033670,0.031368
R&D Spend,-0.143165,0.105711,0.039068,1.000000,0.241955,0.724248,0.972900
Administration,-0.015478,0.010493,0.005145,0.241955,1.000000,-0.032154,0.200717
Marketing Spend,-0.168875,0.205685,-0.033670,0.724248,-0.032154,1.000000,0.747766
Profit,-0.145837,0.116244,0.031368,0.972900,0.200717,0.747766,1.000000


In [12]:
#Guideline by Prashant N
#
# Select those feature columns whose corr is greater than equal to +/-0.5

In [13]:
# Selected Features are: rdSpend and markSpend
# Label: Profit
#
# Build Model and check model satisfies by Genralization Criteria

# 2. Backward Elimination using OLS (Ordinary Least Square)

In [14]:
# preform all in 
allInFeatures = np.append(np.ones((len(finalFeatureSet),1)).astype(int),finalFeatureSet,axis=1)

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [16]:
# decide sl value

SL = 0.05

In [17]:
#Step3: Perform OLS

# endog ---- label column
# exog ----- feature column

import statsmodels.regression.linear_model as stat

olsFormula = stat.OLS(endog=label,exog=allInFeatures.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.945
Method:                 Least Squares   F-statistic:                     169.9
Date:                Wed, 29 Apr 2026   Prob (F-statistic):           1.34e-27
Time:                        19:31:18   Log-Likelihood:                -525.38
No. Observations:                  50   AIC:                             1063.
Df Residuals:                      44   BIC:                             1074.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3.763e+04   5073.636      7.417      0.000    2.74e+04    4.79e+04
x1          1.249e+04   2449.797      5.099      0.000    7554.868    1.74e+04
x2          1.269e+04   2726.700      4.654      0.000    7195.596    1.82e+04
x3          1.245e+04   2486.364      5.007      0.000    7439.285    1.75e+04
x4             0.8060      0.046     17.369      0.000       0.712       0.900
x5            -0.0270      0.052     -0.517      0.608      -0.132       0.078
x6             0.0270      0.017      1.574      0.123      -0.008       0.062
==============================================================================
Omnibus:                       14.782   Durbin-Watson:                   1.283
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.266
Skew:                          -0.948   Prob(JB):                     2.41e-05
Kurtosis:                       5.572   Cond. No.                     2.14e+18
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 8.48e-25. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [18]:
#Step4: Select the feature column that has the highest p-value

# Selected feature is x5(adminSpend) ------ 0.608

In [19]:
#Step5: Check the condition
#
# if pvalue > SL:
#      eliminate feature and recreate new feature set
# else:
#      Go to step7

In [21]:
newFeatureSet =  finalFeatureSet[:,[0,1,2,3,5]]

In [23]:
olsFormula =stat.OLS(endog=label,exog=newFeatureSet.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.950
Model:                            OLS   Adj. R-squared:                  0.946
Method:                 Least Squares   F-statistic:                     215.8
Date:                Wed, 29 Apr 2026   Prob (F-statistic):           9.72e-29
Time:                        19:34:31   Log-Likelihood:                -525.53
No. Observations:                  50   AIC:                             1061.
Df Residuals:                      45   BIC:                             1071.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
x1          4.696e+04   3119.471     15.053      0.000    4.07e+04    5.32e+04
x2           4.71e+04   3670.129     12.833      0.000    3.97e+04    5.45e+04
x3          4.694e+04   3342.591     14.043      0.000    4.02e+04    5.37e+04
x4             0.7967      0.042     18.771      0.000       0.711       0.882
x5             0.0298      0.016      1.842      0.072      -0.003       0.062
==============================================================================
Omnibus:                       14.640   Durbin-Watson:                   1.257
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.037
Skew:                          -0.938   Prob(JB):                     2.70e-05
Kurtosis:                       5.565   Cond. No.                     9.45e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 9.45e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [24]:
newFeatureSet =  finalFeatureSet[:,[0,1,2,3]]

In [25]:
olsFormula = stat.OLS(endog=label,exog=newFeatureSet.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.943
Method:                 Least Squares   F-statistic:                     272.4
Date:                Wed, 29 Apr 2026   Prob (F-statistic):           2.76e-29
Time:                        19:35:52   Log-Likelihood:                -527.35
No. Observations:                  50   AIC:                             1063.
Df Residuals:                      46   BIC:                             1070.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
x1          4.875e+04   3040.118     16.036      0.000    4.26e+04    5.49e+04
x2          4.991e+04   3422.664     14.584      0.000     4.3e+04    5.68e+04
x3          4.876e+04   3275.140     14.888      0.000    4.22e+04    5.54e+04
x4             0.8530      0.030     28.226      0.000       0.792       0.914
==============================================================================
Omnibus:                       13.418   Durbin-Watson:                   1.122
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               17.605
Skew:                          -0.907   Prob(JB):                     0.000150
Kurtosis:                       5.271   Cond. No.                     2.90e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.9e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

# Train ML Model

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [28]:
for rs in range(1,101):
    x_train,x_test,y_train,y_test = train_test_split(newFeatureSet,label,test_size=0.2,random_state=rs)
    model = LinearRegression()
    
    model.fit(x_train,y_train)
    
    trainScore = model.score(x_train,y_train)
    testScore = model.score(x_test,y_test)
    
    if trainScore < testScore >= 0.9:
        print(f"test score {testScore} and train score {trainScore} , rs {rs}")

test score 0.9611102529882832 and train score 0.9387161169637389 , rs 1
test score 0.9775946857818008 and train score 0.9346763047942375 , rs 2
test score 0.9559654175007253 and train score 0.9422463368126369 , rs 3
test score 0.9590556616168858 and train score 0.9424803388022351 , rs 4
test score 0.972476128180872 and train score 0.938481161691229 , rs 5
test score 0.9813828774663522 and train score 0.9360316917181242 , rs 10
test score 0.9525970614685341 and train score 0.9429161309250664 , rs 12
test score 0.9665035917005018 and train score 0.9382536651204462 , rs 14
test score 0.9458552878814185 and train score 0.9448618299320202 , rs 20
test score 0.9598546868412494 and train score 0.9424357627200555 , rs 21
test score 0.9675482978516404 and train score 0.9393665190660465 , rs 22
test score 0.9584704147342656 and train score 0.9432925534944119 , rs 24
test score 0.962818229150605 and train score 0.939518924777711 , rs 26
test score 0.9523021832222983 and train score 0.942907038009

In [30]:
import pickle
x_train,x_test,y_train,y_test = train_test_split(newFeatureSet,label,test_size=0.2,random_state=10)
model = LinearRegression()

model.fit(x_train,y_train)

trainScore = model.score(x_train,y_train)
testScore = model.score(x_test,y_test)

if testScore >=0.9:
    pickle.dump(model,open('model.pkl','wb'))
else:
    print("Retry or Select better random state")

In [32]:
model = pickle.load(open('model.pkl','rb'))

predictProfit = model.predict(np.array([[0,1,0,120000]]))
print(f"Predicted Profit {predictProfit}")

Predicted Profit [152576.49358206]
